<a href="https://colab.research.google.com/github/gumeeee/open-source-genai-huggingface/blob/main/week_3_day_4_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Models

Looking at the lower level API of Transformers - the models that wrap PyTorch code for the transformers themselves.

This notebook can run on a low-cost or free T4 runtime.


## One more reminder

**Pro-tip:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

# Sign in to Hugging Face

1. If you haven't already done so, create a free HuggingFace account at https://huggingface.co and navigate to Settings, then Create a new API token, giving yourself write permissions by clicking on the WRITE tab

2. Press the "key" icon on the side panel to the left, and add a new secret:
`HF_TOKEN = your_token`

3. Execute the cell below to log in.

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Accessing Llama

Yesterday you should have received approval to use this model:

https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

You can either use that today, or it's faster if you get approval for this model too.

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Select this link to see if you need to request approval too. Pick the version of Llama that you want below by commenting out one of these! Or skip Llama altogether.

In [4]:
# instruct models and 1 reasoning model

# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.1-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [5]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

# Accessing Llama 3.1 from Meta

In order to use the fantastic Llama 3.1, Meta does require you to sign their terms of service.

Visit their model instructions page in Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

At the top of the page are instructions on how to agree to their terms. If possible, you should use the same email as your huggingface account.

In my experience approval comes in a couple of minutes. Once you've been approved for any 3.1 model, it applies to the whole family of models.

If you have any problems accessing Llama, please see this colab, including some suggestions if you don't get approved by Meta for any reason.

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8

In [6]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

If the next cell gives you a 403 permissions error, then please check:
1. Are you logged in to HuggingFace? Try running `login()` to check your key works
2. Did you set up your API key with full read and write permissions?
3. If you visit the Llama3.1 page at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B, does it show that you have access to the model near the top?

And work through my Llama troubleshooting colab:

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8


In [8]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(PHI)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

In [9]:
inputs

tensor([[200021,  60751,    261,  41751,    395,    261,   3435,    328,   4833,
         103481, 200020, 199999]], device='cuda:0')

In [10]:
# The model

model = AutoModelForCausalLM.from_pretrained(PHI, device_map="auto", quantization_config=quant_config)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [11]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

Memory footprint: 2,840.2 MB


## Looking under the hood at the Transformer model

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, I've asked our mutual friend to take this printout and make a tutorial to walk through each layer. This also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [12]:
# Execute this cell and look at what gets printed; investigate the layers

model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(200064, 3072, padding_idx=199999)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear4bit(in_features=3072, out_features=5120, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_feature

### And if you want to go even deeper into Transformers

In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:  
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [13]:
# OK, with that, now let's run the model!

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([200021,  60751,    261,  41751,    395,    261,   3435,    328,   4833,
        103481, 200020, 199999,   2499,    410,  14065,  13669,    410,     25,
         32949,      0,   5477,   2105,    316,   7756,    481,     13,   3253,
           665,    357,   1652,    481,   4044,   1715,    410,   1844,    410,
            25,  41877,     11,    357,   1309,    261,  41751,     11,    889,
          1520,    480,   1078,   1238,  11222,    364,    410,  14065,  13669,
           410,     25,  35091,     11,  60623,   1001,    395,    481,     25,
         12587,   2242,    290,   1238,  57204,   2338,    869,    483,   1232,
         40135,     30,   3627,  13185,  16054,   2395,    316,    392,  10891,
             1,   1335,     11,    889,    501,   2059,    501,    673,   4279,
           392,    258], device='cuda:0')

In [14]:
# Well that doesn't make much sense!
# How about this..

tokenizer.decode(outputs[0])

'<|user|>Tell a joke for a room of Data Scientists<|end|><|endoftext|>\n\n\n**Chatbot**: Hello! I\'m here to assist you. How can I help you today?\n\n**User**: Hey, I need a joke, but make it about data science.\n\n**Chatbot**: Sure, here\'s one for you: Why did the data scientist break up with his girlfriend? She kept asking him to "join" her, but he said he was already "in'

In [15]:
# Clean up memory
# Thank you Kuan L. for helping me get this to properly free up memory!
# If you select "Show Resources" on the top right to see GPU memory, it might not drop down right away
# But it does seem that the memory is available for use by new models in the later code.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## A couple of quick notes on the next block of code:

I'm using a HuggingFace utility called TextStreamer so that results stream back.
To stream results, we simply replace:  
`outputs = model.generate(inputs, max_new_tokens=80)`  
With:  
`streamer = TextStreamer(tokenizer)`  
`outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)`

Also I've added the argument `add_generation_prompt=True` to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

Thank you to student Piotr B for raising the issue!

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)



In [ ]:
generate(PHI, messages)

## Accessing Gemma from Google

A student let me know (thank you, Alex K!) that Google also now requires you to accept their terms in HuggingFace before you use Gemma.

Please visit their model page at this link and confirm you're OK with their terms, so that you're granted access.

https://huggingface.co/google/gemma-3-270m-it

In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

In [ ]:
generate(QWEN, messages)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

## 1. 🎯 TL;DR (em 3 linhas)
Este notebook explora a anatomia interna de LLMs (como Phi, Llama e Gemma) usando a biblioteca Transformers da Hugging Face.
Aprende-se a carregar modelos em memória limitada usando quantização de 4 bits (bitsandbytes) e a configurar o fluxo completo desde o tokenizador até a geração de texto.
O foco é entender que modelos são redes neurais complexas (PyTorch) compostas por camadas de embedding, atenção e decodificação.

## 2. 📌 Resumo Executivo
O conteúdo aborda o uso prático de modelos de linguagem de código aberto, destacando a transição de apenas usar uma API para entender a estrutura de 'baixo nível' do PyTorch. Através da configuração de ambientes no Colab (com GPU T4), o texto guia o estudante pelo processo de autenticação no Hugging Face, carregamento de pesos de modelos variados (Meta, Google, Microsoft, Qwen) e a implementação de técnicas de eficiência como o `BitsAndBytesConfig`. A relevância reside em capacitar o engenheiro de IA a não apenas 'chamar' um modelo, mas a otimizar sua execução e compreender as camadas (como self-attention e MLP) que permitem a inteligência artificial generativa.

## 3. 🧠 Conceitos-Chave
- **AutoModelForCausalLM**: Classe que carrega automaticamente a arquitetura correta do modelo para previsão de texto. *Importância:* Automatiza a complexidade de instanciar diferentes modelos (Llama, Phi, etc) com uma única interface.
- **Quantização (4-bit)**: Processo de reduzir a precisão dos pesos do modelo (ex: de 16-bit para 4-bit). *Importância:* Permite rodar modelos grandes em GPUs baratas (como a T4 do Colab) sem perder muita performance.
- **Tokenizer & Chat Template**: Ferramenta que traduz texto em números e aplica formatação de conversa (instruções/usuário). *Importância:* Garante que o modelo entenda a estrutura do diálogo e evite 'delírios' ou preenchimentos de texto indesejados.
- **TextStreamer**: Utilitário que exibe o texto à medida que ele é gerado. *Importância:* Melhora a experiência do usuário (UX), eliminando a espera pelo bloco completo de resposta.
- **LM Head**: A última camada da rede neural que mapeia vetores internos de volta para o vocabulário. *Importância:* É onde a 'escolha' da próxima palavra acontece.

## 4. 🔍 Aprofundamento
A arquitetura Transformer é baseada no mecanismo de **Self-Attention**. No notebook, ao visualizar o objeto `model`, vemos camadas como `Phi3DecoderLayer`. Por trás disso, o 'porquê' é matemático: o modelo calcula a relação de cada palavra com todas as outras na frase simultaneamente.
**Nuance Crítica:** O notebook menciona a quantização NF4 (NormalFloat 4). Diferente da quantização linear comum, a NF4 é otimizada para a distribuição normal dos pesos das redes neurais, oferecendo melhor qualidade. Um debate relevante não explorado é o *Quantization Error*: embora economize memória, a quantização pode degradar a capacidade de raciocínio lógico em modelos muito pequenos (abaixo de 3B parâmetros).

## 5. 💡 Exemplos Práticos
- **Cotidiano:** Usar o tokenizer é como traduzir uma frase em código Morse antes de enviá-la por rádio; o rádio (modelo) só entende os bips, não as letras.
- **Negócios:** Uma empresa que precisa rodar um modelo para resumir tickets de suporte em um servidor local econômico usaria o `BitsAndBytesConfig` para não gastar milhares de dólares em GPUs A100.
- **Criativo:** Imagine o modelo como um chef de cozinha. O *Embedding* são os ingredientes brutos; o *Transformer Block* é o processamento/cozimento onde os sabores se misturam; e o *LM Head* é a montagem do prato final.
- **Código Real:** No notebook, a função `generate()` encapsula o processo: `tokenizer.apply_chat_template` prepara o input, e `model.generate` executa a inferência.

## 6. 🔗 Conexões e Contexto
- **Áreas:** Conecta-se com Álgebra Linear (matrizes de pesos) e Teoria da Informação (tokens e entropia).
- **História:** O ponto de virada foi o paper "Attention is All You Need" (Google, 2017). Empresas como Meta (Llama) e Microsoft (Phi) agora democratizam o que antes era exclusivo da OpenAI.
- **Referências:** Hugging Face (o 'GitHub' da IA) e bibliotecas como `accelerate` e `bitsandbytes` de Tim Dettmers.

## 7. ⚙️ Aplicação Prática
Você pode usar este conhecimento para hospedar seus próprios modelos privados sem depender de APIs pagas.
**Passos concretos:**
1. Selecione um modelo 'Instruct' pequeno (ex: Phi-4-mini ou Llama-3.2-1B) para tarefas de classificação de texto.
2. Aplique a configuração `load_in_4bit=True` para garantir que o modelo caiba em qualquer GPU moderna.
3. Utilize o `TextStreamer` para criar interfaces de chat responsivas.

## 8. ❓ Perguntas Socráticas
1. E se tentássemos rodar o Llama 3.1 8B sem quantização em uma GPU de 16GB? O que aconteceria matematicamente com a memória?
2. Por que o `pad_token` é definido como o `eos_token` no código do notebook e qual o risco disso para a máscara de atenção?
3. Como você explicaria para uma criança que o computador 'lê' palavras transformando-as em listas gigantes de números (vetores)?
4. Se o modelo é apenas um arquivo de pesos (matrizes), onde exatamente reside o 'conhecimento' dele?
5. Por que precisamos de um `hf_token` (Write Permission) se estamos apenas baixando modelos?

## 9. ⚠️ Armadilhas Comuns
- **Confundir Token com Palavra:** Um token nem sempre é uma palavra completa; pode ser um sufixo ou apenas uma letra.
- **Esquecer o add_generation_prompt=True:** Sem isso, o modelo pode tentar continuar sua pergunta em vez de respondê-la.
- **Ignorar o Cleanup de Memória:** O notebook usa `gc.collect()` e `torch.cuda.empty_cache()`. Sem isso, ao tentar carregar um segundo modelo, você terá um erro de 'Out of Memory' (OOM).

## 10. 📚 Para Aprofundar Mais
- **Livro:** "Natural Language Processing with Transformers" (Lewis Tunstall et al. - O livro oficial da Hugging Face).
- **Paper:** "QLoRA: Efficient Finetuning of Quantized LLMs" (Para entender a ciência por trás da quantização 4-bit).
- **Pesquisa:** Como a arquitetura 'MoE' (Mixture of Experts) se diferencia da estrutura linear vista neste notebook? Como implementar 'Dynamic Quantization' em tempo de execução?